# 04 — Baseline & Models

**Workstream:** Price Prediction · **Batch 3 (Day 4)** · **Status: IMPLEMENTED**

Trains naive baseline + Linear Regression + Random Forest + XGBoost (ADR-001) on the 1-day-horizon target,
compares all four on the SAME chronological validation split. This run is on synthetic data — real numbers
land in `ai/experiments.csv` once `real_full` data is used; this notebook's results stay here, clearly
labeled, and are never copied into that file.

In [1]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai" / "common" / "config.py").exists())
sys.path.insert(0, str(_root))
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

train = pd.read_csv(_root / "data/processed/mandi_features_train_synthetic.csv")
val = pd.read_csv(_root / "data/processed/mandi_features_val_synthetic.csv")
print(f"train={len(train)} rows  val={len(val)} rows")

train=7250 rows  val=1050 rows


## 1. Prepare feature matrix (1-day horizon)

In [2]:
FEATURE_COLS = ["day_of_week","day_of_month","month","week_of_year","is_weekend",
                 "lag_1","lag_3","lag_7","lag_14","rolling_mean_7","rolling_std_7",
                 "rolling_mean_14","rolling_std_14","state_code","market_code","commodity_code"]
TARGET = "target_1d"

def prep(df):
    d = df.dropna(subset=FEATURE_COLS + [TARGET])
    return d[FEATURE_COLS], d[TARGET]

Xtr, ytr = prep(train)
Xval, yval = prep(val)
print(f"Usable rows after dropping NaN (early-series lag/rolling warmup): train={len(Xtr)}  val={len(Xval)}")

Usable rows after dropping NaN (early-series lag/rolling warmup): train=6550  val=1050


## 2. Naive baseline (docs/rag/source-policy.md definition — lag_1 carried forward)

In [3]:
naive_pred = Xval["lag_1"]
naive_mae = mean_absolute_error(yval, naive_pred)
naive_rmse = mean_squared_error(yval, naive_pred) ** 0.5
print(f"naive_baseline   MAE={naive_mae:.2f}  RMSE={naive_rmse:.2f}")

naive_baseline   MAE=74.10  RMSE=98.55


## 3. Train candidates (ADR-001)

In [4]:
models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    "xgboost": XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
}

results = {"naive_baseline": {"mae": naive_mae, "rmse": naive_rmse}}
predictions = {"naive_baseline": naive_pred.values}

for name, model in models.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xval)
    mae = mean_absolute_error(yval, pred)
    rmse = mean_squared_error(yval, pred) ** 0.5
    results[name] = {"mae": mae, "rmse": rmse}
    predictions[name] = pred
    print(f"{name:20s}  MAE={mae:7.2f}  RMSE={rmse:7.2f}  beats naive: {mae < naive_mae}")

linear_regression     MAE=  54.41  RMSE=  72.54  beats naive: True


random_forest         MAE=  57.26  RMSE=  77.43  beats naive: True
xgboost               MAE=  56.71  RMSE=  76.14  beats naive: True


## 4. Comparison table

In [5]:
import pandas as pd
results_df = pd.DataFrame(results).T.sort_values("mae")
results_df["beats_naive"] = results_df["mae"] < naive_mae
results_df

,mae,rmse,beats_naive
linear_regression,54.411997,72.538926,True
xgboost,56.708835,76.140675,True
random_forest,57.258897,77.431868,True
naive_baseline,74.096829,98.554364,False


## 5. Save best model + predictions for `05_evaluation_error_analysis.ipynb`

In [6]:
import pickle
from ai.common.config import PROJECT_ROOT

best_name = results_df.index[0]
best_model = models.get(best_name)
print(f"Best model on validation MAE: {best_name}")

MODEL_DIR = PROJECT_ROOT / "ai" / "price_prediction" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
if best_model is not None:
    with open(MODEL_DIR / f"{best_name}_1d_SYNTHETIC.pkl", "wb") as f:
        pickle.dump(best_model, f)
    print(f"Saved -> {MODEL_DIR / (best_name + '_1d_SYNTHETIC.pkl')}")

import json as _json
with open(_root / "data/processed/model_comparison_1d_synthetic.json", "w") as f:
    _json.dump(results, f, indent=1)
print("Saved comparison metrics -> data/processed/model_comparison_1d_synthetic.json")

Best model on validation MAE: linear_regression
Saved -> /home/claude/AgriConnect/ai/price_prediction/models/linear_regression_1d_SYNTHETIC.pkl
Saved comparison metrics -> data/processed/model_comparison_1d_synthetic.json


## Next: `05_evaluation_error_analysis.ipynb`

**Status: IMPLEMENTED.** All four models trained and compared for real above — this is not a plan, it's a
completed run (synthetic data, clearly labeled throughout).